# 3. Non-Parametric Cosmic-Ray Inverse Reconstruction (`GPProduction` & IFT `LogGaussianField`)

This tutorial replaces `docs/notebooks/03_InverseSolver.ipynb`, in which `ticktack 1.x` inverted the 28-ring `Miyake12_Cedar` record on `Guttler15` via unregularized annual step Monte Carlo (`sf.MC_reconstruct(iters=1000)`) in **`11.6 s`**. Differentiating noisy annual $\Delta^{14}\mathrm{C}$ data without spectral regularization amplified high-frequency measurement noise into wild year-to-year oscillations and unphysical **negative** cosmic-ray production excursions down to **`-1.73 atoms cm^-2 s^-1`**.

`ticktack 2.0` provides two complementary spectral GP basis representations for non-parametric cosmic-ray inversion: causal [Solin & Särkkä (2020)](https://doi.org/10.1007/s11222-019-09886-w) Hilbert-space POD `GPProduction` (initialized from the exact stationary [Smith (1968)](https://doi.org/10.1137/0116017) doubling pre-history covariance and solved in closed form via `linear_posterior` in milliseconds) and reflecting-boundary orthonormal DCT-II Information Field Theory ([Enßlin et al. 2009](https://doi.org/10.1103/PhysRevD.80.105005); `LogGaussianField` + empirical-Bayes `select_gp_kernel`), which guarantees strictly positive cosmic-ray production everywhere while ranking candidate smoothness priors via Laplace marginal evidence and auditing measurement noise via [MacKay (1992)](https://doi.org/10.1162/neco.1992.4.3.415)'s `error_scale` diagnostic.

In [ ]:
import time, jax, jax.numpy as jnp, numpy as np, tinygp.kernels as tk
import ticktack as tt

cedar = tt.TreeRingSeries.from_csv("notebooks/miyake12.csv", hemisphere="north", growth_season="april-september", name="Miyake12_Cedar")
ds, cbm, t_fine = tt.Dataset(cedar, name="Miyake12_Inverse"), tt.load_preset("Guttler15"), jnp.linspace(760.0, 788.0, 225)
gp_model = tt.RadiocarbonModel(cbm, tt.SteadyProduction() + tt.GPProduction(tk.quasisep.Matern32(scale=2.0, sigma=2.2), n_modes=64))
cfg_gp = tt.LinearMarginalConfig(offset_mode="shared", marginalize_gp=True)
t0 = time.perf_counter()
lp_gp = tt.linear_posterior(gp_model, ds, cfg_gp, t_fine=t_fine, n_draws=200, rng_key=jax.random.PRNGKey(301))
jax.block_until_ready(lp_gp.q_fine_mean)
dt_gp_cold = time.perf_counter() - t0
t1 = time.perf_counter()
lp_gp_w = tt.linear_posterior(gp_model, ds, cfg_gp, t_fine=t_fine, n_draws=200, rng_key=jax.random.PRNGKey(302))
jax.block_until_ready(lp_gp_w.q_fine_mean)
print(f"Closed-form GPProduction: cold={dt_gp_cold*1e3:.0f} ms, warm={(time.perf_counter()-t1)*1e3:.2f} ms | min 95% band Q={float(jnp.percentile(lp_gp.q_fine_draws, 2.5)):.2f} atoms/cm^2/s")

## Strictly Positive IFT Reconstruction & Automatic Kernel Selection (`select_gp_kernel`)

While closed-form linear GP inversion runs in milliseconds, a stationary symmetric Gaussian prior broadens across the 774 CE impulse and still permits negative lower credible bounds in quiet baseline decades. Passing `field_kind="log_gaussian"` to `select_gp_kernel` parameterizes strictly positive production on a reflecting-boundary DCT-II basis with mean-preserving Itô variance correction:

$$Q(t) = Q_{\mathrm{std}}\exp\!\left[f(t) - \tfrac{1}{2}\sigma_f^2(t)\right]$$

This optimizes spectral hyperparameters across candidate Matérn and Squared-Exponential kernels faster than `v1`'s `11.6 s` `MC_reconstruct` while enforcing positivity identically.

In [ ]:
cands = ("matern12", "matern32", "matern52", "squared_exp")
t0 = time.perf_counter()
sel = tt.select_gp_kernel(cbm, ds, candidates=cands, field_kind="log_gaussian", n_grid=80, pad_years=10.0, multi_start=1, rng_key=jax.random.PRNGKey(303))
jax.block_until_ready(sel.posterior_q)
dt_ift_cold = time.perf_counter() - t0
t1 = time.perf_counter()
sel_w = tt.select_gp_kernel(cbm, ds, candidates=cands, field_kind="log_gaussian", n_grid=80, pad_years=10.0, multi_start=1, rng_key=jax.random.PRNGKey(304))
jax.block_until_ready(sel_w.posterior_q)
dt_ift_warm = time.perf_counter() - t1
q_min, q_peak = float(jnp.min(sel.posterior_q)), float(jnp.max(jnp.median(sel.posterior_q, axis=0)))
print(f"{sel.format_table()}\nmin posterior Q = +{q_min:.3f} atoms/cm^2/s (v1 MC_reconstruct: -1.73), peak median Q = {q_peak:.2f}\nTiming vs v1 (11.6 s): cold={dt_ift_cold:.2f} s ({11.6/dt_ift_cold:.1f}x), warm={dt_ift_warm:.2f} s ({11.6/dt_ift_warm:.1f}x)")

## Positivity-Preserving Cosmic-Ray Production Posterior $Q(t)$

The Laplace posterior draws from the highest-evidence `LogGaussianField` kernel localize the impulsive 774–775 CE production burst cleanly above the pre-industrial reference rate $Q_{\mathrm{std}} = 1.76\,\mathrm{atoms\,cm^{-2}\,s^{-1}}$ while keeping every posterior trajectory strictly positive across the 760–787 CE baseline.

In [ ]:
in_win = (sel.t_grid >= 759.5) & (sel.t_grid <= 788.0)
fig_q, ax_q = tt.plot_production_posterior(
    sel.t_grid[in_win],
    sel.posterior_q[:, in_win],
    q_std=float(tt.DEFAULT_Q_STD),
    n_spaghetti=10,
    title=f"Strictly Positive IFT LogGaussianField Reconstruction ({sel.best_name})",
)

## Forward $\Delta^{14}\mathrm{C}(t)$ Trajectory and Standardized Pulls

Unlike `v1` `InverseSolver`, which forced users to hand-write a forward ODE wrapper anchored to `d14c_data[0]`, the fitted `RadiocarbonModel` predicts continuous and ring-binned $\Delta^{14}\mathrm{C}$ directly with marginalized baseline offset `global_offset`.

In [ ]:
fit_ift = sel.best_result.fit
d14c_ift_fine = fit_ift.model.predict_fine(t_fine, hemisphere="north") + fit_ift.global_offset
fig_d, axes_d = tt.plot_d14c_fit(
    ds,
    t_fine=t_fine,
    d14c_pred=d14c_ift_fine,
    ring_pred=fit_ift.d14c_pred,
    show_residuals=True,
    title="Miyake12 Cedar Data Fit & Pulls from Non-Parametric LogGaussianField",
)

## One-to-One Observed vs Reconstructed Calibration

Inspecting `plot_data_model_correlation` verifies that the regularized `LogGaussianField` inverse solution avoids overfitting individual noisy rings (`WRMS pull < 1`) while tracking the `~20‰` Miyake jump without systematic bias.

In [ ]:
fig_corr, ax_corr = tt.plot_data_model_correlation(
    ds,
    fit_ift.d14c_pred,
    title="Non-Parametric IFT Reconstruction: 1:1 Data–Model Correlation",
)

Combining closed-form Solin–Särkkä `GPProduction` (millisecond Wiener filtering) with empirical-Bayes `LogGaussianField` reconstruction replaces `v1`'s `11.6 s` unregularized step-inversion—and its unphysical `-1.73 atoms cm^-2 s^-1` negative rates—with strictly positive, evidence-ranked cosmic-ray histories.